### NB_OTT_BronzeToSilver
Objetivo:
Integrar los tickets procedentes de dos mecanismos de ingesta:
* Batch       -> Bronze.Tickets
* Streaming   -> Bronze.KafkaTickets
y generar una representación limpia, normalizada y deduplicada en: Silver.Tickets
Este notebook forma parte del pipeline operacional.

Principales responsabilidades:
* Homogeneizar batch y streaming.
* Normalizar tipos.
* Resolver duplicados entre ambas fuentes.
* Aplicar controles básicos de calidad.
* Preparar text_for_embedding.
* Añadir metadatos de procesamiento.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
# ============================================================
# 1. Carga de tickets batch
#
# Los tickets cargados mediante el flujo batch se encuentran en Bronze.Tickets.
#
# ingestion_source identifica el mecanismo de ingesta y se conserva posteriormente en Silver para trazabilidad.
# ============================================================

df_batch = (
    spark.table("Bronze.Tickets")

    # Normalización de timestamps.
    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    )
    .withColumn(
        "resolved_at",
        F.to_timestamp("resolved_at")
    )

    # Identificación del mecanismo de ingesta.
    .withColumn(
        "ingestion_source",
        F.lit("batch")
    )
)

batch_count = df_batch.count()

print(
    f"Batch tickets loaded: {batch_count}"
)

In [ ]:
# ============================================================
# 2. Carga de tickets streaming
#
# Los eventos procedentes de Confluent Kafka son consumidos por Microsoft Fabric Eventstream y persistidos en:
#
#   Bronze.KafkaTickets
#
# Se aplica la misma normalización de tipos que al flujo batch.
# ============================================================

df_stream = (
    spark.table("Bronze.KafkaTickets")

    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    )
    .withColumn(
        "resolved_at",
        F.to_timestamp("resolved_at")
    )

    .withColumn(
        "ingestion_source",
        F.lit("kafka")
    )
)

stream_count = df_stream.count()

print(
    f"Kafka tickets loaded: {stream_count}"
)

In [ ]:
# ============================================================
# 3. Integración de batch y streaming
#
# allowMissingColumns=True permite integrar ambas fuentes aun cuando exista alguna diferencia menor de esquema entre ellas.
#
# El resultado representa la entrada conjunta antes de aplicar reglas de deduplicación y calidad.
# ============================================================

df_combined = (
    df_batch
    .unionByName(
        df_stream,
        allowMissingColumns=True
    )
)

combined_count = df_combined.count()

print("Input integration")
print("-----------------")
print("Batch:", batch_count)
print("Kafka:", stream_count)
print("Combined:", combined_count)


assert combined_count == (
    batch_count + stream_count
), (
    "Unexpected record count after batch/stream union."
)

In [ ]:
# ============================================================
# 3. Integración de batch y streaming
#
# allowMissingColumns=True permite integrar ambas fuentes aun cuando exista alguna diferencia menor de esquema entre ellas.
#
# El resultado representa la entrada conjunta antes de aplicar reglas de deduplicación y calidad.
# ============================================================

df_combined = (
    df_batch
    .unionByName(
        df_stream,
        allowMissingColumns=True
    )
)

combined_count = df_combined.count()

print("Input integration")
print("-----------------")
print("Batch:", batch_count)
print("Kafka:", stream_count)
print("Combined:", combined_count)


assert combined_count == (
    batch_count + stream_count
), (
    "Unexpected record count after batch/stream union."
)

In [ ]:
# ============================================================
# 4. Validación básica de datos de entrada
#
# Antes de transformar Silver calculamos indicadores básicos que posteriormente permiten medir qué registros han sido descartados o deduplicados.
# ============================================================

unique_input_ids = (
    df_combined
    .select("ticket_id")
    .filter(
        F.col("ticket_id").isNotNull()
    )
    .distinct()
    .count()
)

null_ticket_ids = (
    df_combined
    .filter(
        F.col("ticket_id").isNull()
    )
    .count()
)

null_descriptions = (
    df_combined
    .filter(
        F.col("description").isNull()
    )
    .count()
)

print("Input quality")
print("-------------")
print(
    "Total records:",
    combined_count
)
print(
    "Unique non-null ticket IDs:",
    unique_input_ids
)
print(
    "Null ticket IDs:",
    null_ticket_ids
)
print(
    "Null descriptions:",
    null_descriptions
)

In [ ]:
# ============================================================
# 5. Deduplicación entre mecanismos de ingesta
#
# Un mismo ticket puede aparecer:
#
# - en la carga batch histórica;
# - y posteriormente a través de Kafka.
#
# Cuando existe el mismo ticket_id en ambas fuentes, se da prioridad a la versión procedente de Kafka.
#
# Motivo:
# la versión recibida mediante streaming se considera la representación más reciente dentro del flujo operacional.
# ============================================================

dedup_window = (
    Window
    .partitionBy(
        "ticket_id"
    )
    .orderBy(
        F.when(
            F.col(
                "ingestion_source"
            ) == "kafka",
            F.lit(1)
        ).otherwise(
            F.lit(2)
        )
    )
)

df_deduplicated = (
    df_combined

    .withColumn(
        "_row_number",
        F.row_number().over(
            dedup_window
        )
    )

    .filter(
        F.col("_row_number") == 1
    )

    .drop(
        "_row_number"
    )
)

In [ ]:
# ============================================================
# 6. Reglas de calidad y normalización
#
# Se eliminan registros que no pueden ser procesados adecuadamente:
#
# - ticket_id nulo
# - description nula
#
# Posteriormente se normalizan los principales campos textuales utilizados tanto para reporting como para el procesamiento semántico.
# ============================================================

df_silver = (
    df_deduplicated

    # ----------------------------------------
    # Registros obligatorios
    # ----------------------------------------

    .filter(
        F.col("ticket_id").isNotNull()
    )

    .filter(
        F.col("description").isNotNull()
    )

    # ----------------------------------------
    # Limpieza de título y descripción
    # ----------------------------------------

    .withColumn(
        "title",
        F.trim(
            F.col("title")
        )
    )

    .withColumn(
        "description",
        F.trim(
            F.col("description")
        )
    )

    # ----------------------------------------
    # Normalización de categoría
    # ----------------------------------------

    .withColumn(
        "category",
        F.initcap(
            F.trim(
                F.col("category")
            )
        )
    )

    # ----------------------------------------
    # Normalización de prioridad
    # ----------------------------------------

    .withColumn(
        "priority",
        F.initcap(
            F.trim(
                F.col("priority")
            )
        )
    )
)

In [ ]:
# ============================================================
# 7. Construcción de text_for_embedding
#
# Para la representación semántica se combinan:
#
#   title + description
#
# y se normaliza el texto a minúsculas.
#
# Este campo será consumido posteriormente por:
#
#   NB_OTT_GenerateEmbeddings
#
# No se utiliza incident_id ni ninguna otra información del
# ground truth para generar esta representación.
# ============================================================

df_silver = (
    df_silver
    .withColumn(
        "text_for_embedding",
        F.lower(
            F.concat_ws(
                " ",
                F.col("title"),
                F.col("description")
            )
        )
    )
)

In [ ]:
# ============================================================
# 8. Metadatos de procesamiento
#
# ingestion_source:
#     indica cómo llegó físicamente el ticket:
#       - batch
#       - kafka
#
# _source:
#     identifica el origen lógico del dataset utilizado en
#     este prototipo.
#
# _processed_at:
#     registra cuándo se produjo la transformación Silver.
# ============================================================

df_silver = (
    df_silver

    .withColumn(
        "_processed_at",
        F.current_timestamp()
    )

    .withColumn(
        "_source",
        F.lit(
            "synthetic_ott_tickets"
        )
    )
)

In [ ]:
# ============================================================
# 9. Data Quality Checks
#
# Antes de escribir Silver verificamos:
#
# - ticket_id único;
# - ausencia de ticket_id nulos;
# - ausencia de descriptions nulas;
# - existencia de text_for_embedding;
# - presencia de ambas fuentes cuando Kafka contiene datos.
# ============================================================

silver_count = (
    df_silver.count()
)

duplicate_ticket_ids = (
    df_silver
    .groupBy(
        "ticket_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

silver_null_ids = (
    df_silver
    .filter(
        F.col("ticket_id").isNull()
    )
    .count()
)

silver_null_descriptions = (
    df_silver
    .filter(
        F.col("description").isNull()
    )
    .count()
)

empty_embedding_texts = (
    df_silver
    .filter(
        F.col(
            "text_for_embedding"
        ).isNull()
        |
        (
            F.length(
                F.trim(
                    F.col(
                        "text_for_embedding"
                    )
                )
            ) == 0
        )
    )
    .count()
)


assert duplicate_ticket_ids == 0, (
    f"Found {duplicate_ticket_ids} duplicated ticket IDs."
)

assert silver_null_ids == 0, (
    f"Found {silver_null_ids} null ticket IDs."
)

assert silver_null_descriptions == 0, (
    f"Found {silver_null_descriptions} null descriptions."
)

assert empty_embedding_texts == 0, (
    f"Found {empty_embedding_texts} empty embedding texts."
)

In [ ]:
# ============================================================
# 10. Validación de mecanismos de ingesta
# ============================================================

source_counts = {
    row["ingestion_source"]:
        row["count"]

    for row in (
        df_silver
        .groupBy(
            "ingestion_source"
        )
        .count()
        .collect()
    )
}

silver_batch_count = (
    source_counts.get(
        "batch",
        0
    )
)

silver_kafka_count = (
    source_counts.get(
        "kafka",
        0
    )
)

print("Silver ingestion sources")
print("------------------------")
print(
    "Batch:",
    silver_batch_count
)

print(
    "Kafka:",
    silver_kafka_count
)

if stream_count > 0:

    assert silver_kafka_count > 0, (
        "Kafka tickets were loaded but none survived "
        "the Silver transformation."
    )

In [ ]:
# ============================================================
# 11. Persistencia en Silver
#
# Para el TFM se reconstruye Silver.Tickets mediante overwrite.
#
# Esta decisión simplifica la reproducibilidad del pipeline. Una implementación productiva podría evolucionar hacia procesamiento incremental mediante Delta MERGE.
# ============================================================

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS Silver"
)

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "Silver.Tickets"
    )
)

print(
    "Silver.Tickets written successfully."
)

In [ ]:
# ============================================================
# 12. Validación posterior a persistencia
# ============================================================

df_saved = spark.table(
    "Silver.Tickets"
)

saved_count = (
    df_saved.count()
)

assert saved_count == silver_count, (
    f"Expected {silver_count} Silver tickets "
    f"but persisted {saved_count}."
)


saved_duplicates = (
    df_saved
    .groupBy(
        "ticket_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

assert saved_duplicates == 0, (
    "Duplicated ticket IDs detected after persistence."
)


print("Bronze → Silver completed")
print("-------------------------")
print(
    "Input records:",
    combined_count
)

print(
    "Silver records:",
    saved_count
)

print(
    "Batch in Silver:",
    silver_batch_count
)

print(
    "Kafka in Silver:",
    silver_kafka_count
)

print(
    "Duplicates:",
    saved_duplicates
)

print(
    "\nNB_OTT_BronzeToSilver "
    "completed successfully."
)